# CrocoLake Temperature Map - North West Atlantic

This notebook shows how to read and visualize CrocoLake data stored in parquet format, creating a map of temperature measurements in the North West Atlantic.

CrocoLake contains QCed data from different datasets (Argo, GLODAP, Spray Gliders). We use the `ArgoData.jl` package (Forget, G., & collaborators) to read the parquet files.

- Milanese, E., & Nicholson, D. (2025). Sample parquet datasets of Argo program ocean data [Data set]. Zenodo. https://doi.org/10.5281/zenodo.15198578
- https://github.com/boom-lab/argo2parquet-public
- https://github.com/boom-lab/crocolaketools-public
- https://euroargodev.github.io/ArgoData.jl/dev/

## Activate Environment + Packages

- `ArgoData.jl` provides the parquet reading functions
- `CairoMakie.jl` is the plotting library

In [ ]:
using Pkg; Pkg.activate(".")
Pkg.add("DataFrames")
Pkg.add("Dates")
Pkg.add("Statistics")

In [ ]:
Pkg.status()

In [ ]:
using ArgoData, CairoMakie, DataFrames, Statistics, Dates
import Glob, Tables

## Download CrocoLake Data

If you haven't already downloaded the CrocoLake PHY dataset, use the command line:

```bash
julia --project=. download_db.jl -d CrocoLake -t PHY --destination ./CrocoLake
```

Note: the download might take a while, the PHY dataset is ~20GB.

## Load CrocoLake Dataset

In [ ]:
# Path to CrocoLake PHY data
folder_pq = "../../crocolaketools/demo/parquet/demo_CROCOLAKE_PHY/"

# Load dataset
da = Argo_parquet.Dataset(folder_pq)

## Extract Subset for North West Atlantic (2010-2019)

We'll filter the data by geographical region, time period, and select only temperature measurements.

In [ ]:
# Define region and time period
lons = -85 .. -60
lats = 30 .. 45
dates = Argo_parquet.DateTime("2010-01-01T00:00:00") .. Argo_parquet.DateTime("2019-12-31T23:59:59")
variables = (:JULD, :LATITUDE, :LONGITUDE, :TEMP, :DB_NAME)

# Extract subset
df1 = Argo_parquet.get_subset_region(da.Dataset, 
    lons=lons, lats=lats, dates=dates, variables=variables)

println("Number of measurements: ", nrow(df1))

## Aggregate Data by Location and Source

Calculate average temperature at each location (binned to 0.5 degree resolution) for each data source.

In [ ]:
using DataFrames

# Remove missing temperature values
df = dropmissing(df1, :TEMP)

# Calculate average temperature by location and source
grouped = combine(groupby(df, [:LATITUDE, :LONGITUDE, :DB_NAME]), 
    :TEMP => Statistics.mean => :temp_avg)

println("Number of spatial bins: ", nrow(grouped))
first(grouped, 10)

## Create Temperature Map

Plot the average temperature in the North West Atlantic, color-coded by data source.

In [ ]:
# Calculate colorbar limits (5th and 95th percentiles)
temps = grouped.temp_avg
cbar_min = quantile(temps, 0.05)
cbar_max = quantile(temps, 0.95)

println("Temperature range: ", round(cbar_min, digits=2), " to ", round(cbar_max, digits=2), " °C")

# Create figure
fig = Figure(size = (1400, 700))
ax = Axis(fig[1, 1],
    xlabel = "Longitude",
    ylabel = "Latitude",
    title = "North-West Atlantic Average Temperature 2010-2019 (°C)",
    aspect = DataAspect())

xlims!(ax, -85, -60)
ylims!(ax, 30, 45)

# Define colors for each source
source_colors = Dict(
    "argo" => :Greens,
    "glodap" => :Purples,
    "oleanderxbt" => :Oranges,
    "spraygliders" => :Blues
)

# Get unique sources
sources = sort(unique(skipmissing(grouped.DB_NAME)))

# Plot each data source
for (i, src_raw) in enumerate(sources)
    source = lowercase(strip(string(src_raw)))
    cmap = get(source_colors, source, :viridis)
    # Filter using normalized DB_NAME
    source_data = grouped[.!ismissing.(grouped.DB_NAME) .&
        (lowercase.(strip.(string.(grouped.DB_NAME))) .== source), :]
    if nrow(source_data) > 0
        scatter!(ax, source_data.LONGITUDE, source_data.LATITUDE,
            color = source_data.temp_avg,
            colormap = cmap,
            colorrange = (cbar_min, cbar_max),
            markersize = 10)

        # Add colorbar
        Colorbar(fig[1, i+1], 
            limits = (cbar_min, cbar_max),
            colormap = cmap,
            label = "Avg TEMP ($(src_raw)) [°C]")
    end
end

fig

## Summary Statistics by Data Source

In [ ]:
stats = combine(groupby(grouped, :DB_NAME),
    :temp_avg => length => :n_bins,
    :temp_avg => Statistics.mean => :mean_temp,
    :temp_avg => Statistics.std => :std_temp,
    :temp_avg => minimum => :min_temp,
    :temp_avg => maximum => :max_temp
)

stats

## Suggested Exercises

Try modifying this notebook to:
- Filter by different time periods or regions
- Map a different parameter (e.g., salinity: `:PSAL`)
- Add pressure filtering to `variables` (e.g., include `:PRES` and filter for surface data)
- Create time series or depth profile plots
- Compare different regions

If you encounter any issues, please [reach out](mailto:enrico.milanese@whoi.edu)!